# 04. Broadcasting Rules & Matrix Centering: Beginner Guide

### 🌟 What Are Broadcasting Rules & Matrix Centering in NumPy?
**Broadcasting** describes how NumPy performs arithmetic between arrays of different shapes (e.g. subtracting a 1D mean from a 2D matrix) without making redundant memory copies, making matrix normalization and feature centering effortless.

This interactive guide loads and works directly with `data/raw_transactions.csv`, giving you real-world hands-on practice.

### 📚 Key Concepts Covered in this Notebook:
- **Broadcasting Compatibility Rules**: Trailing dimensions must be equal, OR one of the dimensions must equal 1.
- **Matrix Centering**: Subtracting 1D column/row mean vectors from 2D matrices without copying data.
- **Dimension Injection**: Covers `np.newaxis` and `None`.


In [1]:
# Setup imports & dataset loading from raw_transactions.csv
import numpy as np
import pandas as pd
import sys
import time
import os

# Load raw transactions and extract aligned NumPy numeric arrays
csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
raw_df = pd.read_csv(csv_path)
clean_raw = raw_df.dropna(subset=['transaction_amount', 'is_fraud', 'account_age_months']).reset_index(drop=True)
amounts = clean_raw['transaction_amount'].to_numpy(dtype=np.float64)
fraud_flags = clean_raw['is_fraud'].to_numpy(dtype=np.int8)
account_ages = clean_raw['account_age_months'].to_numpy(dtype=np.float32)

print(f"NumPy Version: {np.__version__}")
print(f"Loaded from {csv_path} ({len(amounts)} clean aligned rows):")
print(f"- amounts array: shape {amounts.shape}, dtype {amounts.dtype}")
print(f"- fraud_flags array: shape {fraud_flags.shape}, dtype {fraud_flags.dtype}")
print(f"- account_ages array: shape {account_ages.shape}, dtype {account_ages.dtype}")

NumPy Version: 1.26.4
Loaded from ../data/raw_transactions.csv (14262 clean aligned rows):
- amounts array: shape (14262,), dtype float64
- fraud_flags array: shape (14262,), dtype int8
- account_ages array: shape (14262,), dtype float32


### 🔹 Trailing Dimension Compatibility Rules
Broadcasting 1D feature mean vector across 2D transaction matrix. NumPy runs in compiled C memory buffers, enabling mathematical operations across millions of numbers simultaneously in milliseconds. **Tip:** NumPy operations are optimized for homogeneous numeric data, offering massive speed improvements over standard Python loops.

**Syntax:** `tx_mat (N, D) - mean_vec (D,)`


In [2]:
tx_mat = np.column_stack([amounts[:1000], account_ages[:1000]])
col_means = tx_mat.mean(axis=0)
centered_mat = tx_mat - col_means  # (1000, 2) - (2,) -> Broadcasts across rows
print('Matrix Shape:', tx_mat.shape, 'Mean Vector Shape:', col_means.shape)
print('Centered Matrix Head:\n', centered_mat[:3].round(2))

Matrix Shape: (1000, 2) Mean Vector Shape: (2,)
Centered Matrix Head:
 [[ 206.65   -4.1 ]
 [-684.69   51.9 ]
 [-873.02    7.9 ]]


### 🔹 Feature Normalization (Z-Score) via Broadcasting
Normalizes multi-feature transaction matrix to mean 0 and std 1. Converting raw input data (like text from CSVs or APIs) into proper numeric, boolean, or structured types is essential for analysis. **Tip:** Two dimensions are compatible for broadcasting if they are equal, or if one of them is 1. Dimensions are matched from right to left.

**Syntax:** `(X - X.mean(axis=0)) / X.std(axis=0)`


In [3]:
col_stds = tx_mat.std(axis=0)
z_scores = (tx_mat - col_means) / col_stds
print('Z-Scored Feature Columns (Head):\n', z_scores[:3].round(2))

Z-Scored Feature Columns (Head):
 [[ 0.36 -0.12]
 [-1.2   1.47]
 [-1.53  0.22]]


### 🔹 Dimension Expansion with `np.newaxis`
Converts 1D amount vector into column vector `(N, 1)` for outer broadcast. NumPy runs in compiled C memory buffers, enabling mathematical operations across millions of numbers simultaneously in milliseconds. **Tip:** NumPy & Pandas Axis Rule: `axis=0` operates along rows (downwards), while `axis=1` operates along columns (horizontally).

**Syntax:** `amounts[:5, np.newaxis]`


In [4]:
col_vector = amounts[:5, np.newaxis]
print('1D Amounts converted to Column Vector Shape:', col_vector.shape)

1D Amounts converted to Column Vector Shape: (5, 1)


### 🔹 Dimension Expansion with `None`
Aliases `None` to add batch dimension `(1, N)`. NumPy runs in compiled C memory buffers, enabling mathematical operations across millions of numbers simultaneously in milliseconds. **Tip:** NumPy operations are optimized for homogeneous numeric data, offering massive speed improvements over standard Python loops.

**Syntax:** `amounts[:5][None, :]`


In [5]:
row_vector = amounts[:5][None, :]
print('1D Amounts converted to Row Vector Shape:', row_vector.shape)

1D Amounts converted to Row Vector Shape: (1, 5)


## 💡 Real-World Practice & Scenarios
Practical scenarios and common data questions explained simply with real examples.


### 🔍 Scenario: Q1: Softmax Implementation on Risk Logits

**Approach:** Vectorize softmax probability estimation for multi-class transaction classification.
**Syntax:** `np.exp(logits - np.max(logits, axis=1, keepdims=True))`


In [6]:
mock_logits = np.column_stack([amounts[:5]/100, account_ages[:5]/10])
shifted = mock_logits - np.max(mock_logits, axis=1, keepdims=True)
probs = np.exp(shifted) / np.sum(np.exp(shifted), axis=1, keepdims=True)
print('Softmax Output Probabilities:\n', probs.round(3))

Softmax Output Probabilities:
 [[0.999 0.001]
 [0.    1.   ]
 [0.004 0.996]
 [0.023 0.977]
 [0.963 0.037]]
